In [1]:

import os
import sys
import argparse
import textwrap
import importlib
from pathlib import Path


In [2]:
GREEN  = "\033[92m"
YELLOW = "\033[93m"
CYAN   = "\033[96m"
RED    = "\033[91m"
BOLD   = "\033[1m"
RESET  = "\033[0m"


def c(colour, text):
    return f"{colour}{text}{RESET}"

def banner(title: str):
    w = 60
    print(f"\n{c(BOLD, '='*w)}")
    print(c(BOLD, f"  {title}"))
    print(c(BOLD, '='*w))


In [ ]:
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = ''
llm = ChatGroq(model="llama-3.1-8b-instant")

### Test Suite

In [4]:
class TestSuite:
    def __init__(self):
        self.results: list[dict] = []

    def record(self, name: str, passed: bool, detail: str = ""):
        self.results.append({"name": name, "passed": passed, "detail": detail})
        icon = c(GREEN, "✔ PASS") if passed else c(RED, "✗ FAIL")
        print(f"  {icon}  {name}")
        if detail:
            print(f"          {c(YELLOW, detail)}")

    def summary(self):
        total  = len(self.results)
        passed = sum(1 for r in self.results if r["passed"])
        print(c(BOLD, f"\n{'─'*60}"))
        print(c(BOLD, f"  RESULTS: {passed}/{total} tests passed"))
        print(c(BOLD, f"{'─'*60}"))
        if passed < total:
            print(c(RED, "\n  Failed tests:"))
            for r in self.results:
                if not r["passed"]:
                    print(c(RED, f"    ✗ {r['name']}: {r['detail']}"))
        return passed == total
    
suite = TestSuite()

In [5]:
sys.path.append("/mnt/e/Personal/Samarth/repository/skills_with_langchain/Dynamic-Skill-Creation/src/handler")
from skills_registry import get_registry 
def test_registry_loads(suite: TestSuite):
    """Verify the skills registry loads at least one skill."""
    banner("TEST: Registry loads")
    
    registry = get_registry()
    print(f"  Loaded skills: {list(registry.keys())}")
    suite.record(
        "Registry loads skills",
        len(registry) > 0,
        f"{len(registry)} skill(s) found" if registry else "No skills found",
    )
    return registry

test_registry_loads(suite)


  TEST: Registry loads
  Loaded skills: ['browser-search', 'web-page-scraper']
  ✔ PASS  Registry loads skills
          2 skill(s) found


{'browser-search': {'name': 'browser-search',
  'description': 'Use this skill whenever the user needs to find information on the internet,research a topic, look up current events, verify facts, or fetch and read the content of a specific URL. Triggers include: "search the web for",',
  'skill_md_path': PosixPath('/mnt/e/Personal/Samarth/repository/skills_with_langchain/Dynamic-Skill-Creation/src/handler/../skills/browser-search/SKILL.md'),
  'scripts_dir': PosixPath('/mnt/e/Personal/Samarth/repository/skills_with_langchain/Dynamic-Skill-Creation/src/handler/../skills/browser-search/scripts'),
  'full_instructions': '# Browser Search Skill\n\nA two-stage internet research pipeline:\n1. **SerpAPI** — discovers relevant URLs from a Google search\n2. **Firecrawl** — scrapes the top URL(s) and returns clean markdown\n\n---\n\n## Environment Variables\n\n| Variable           | Required | Description                          |\n|--------------------|----------|-------------------------------

In [10]:
sys.path.append("/mnt/e/Personal/Samarth/repository/skills_with_langchain/Dynamic-Skill-Creation/src/agent")

from skill_agent import run_agent

def test_list_skills(suite: TestSuite):
    """Agent can list available skills."""
    banner("TEST: List available skills")
    
    registry = get_registry()
    result   = run_agent(llm=llm,user_query="What skills do you have ?",
                          verbose=False, registry=registry)
    response = result["response"]
    passed   = len(response) > 20
    suite.record("List skills response", passed,
                 response[:120] if not passed else "")
    return response

print(test_list_skills(suite))


  TEST: List available skills
Structured response from is_skill_query_chain: {
  "result": true
}
  ✔ PASS  List skills response
## 🧠 Available Skills

### 1. Web Search 🌐
Use this skill whenever the user needs to find information on the internet,research a topic, look up current events, verify facts, or fetch and read the content of a specific URL.

### 2. Web Page Scraper
Trigger this skill IMMEDIATELY when the user asks to "scrape", "read", "extract text", or "get content" from a specific URL or a general search topic.

---
_Simply describe what you need and I will automatically use the right skill._
